# Adaptive Query Routing for Production RAG

In real-world RAG applications, **one retrieval strategy does not fit all queries**. A short abstract concept requires **HyDE**, an exact version code or SKU needs **BM25**, a comparative question demands **Query Decomposition**, and standard open-domain questions are best handled by **Hybrid Search**.

**Adaptive Query Routing** introduces an intelligent decision engine at runtime that inspects incoming queries, classifies their structural intent, and dynamically dispatches the query to the optimal retrieval engine before final answer synthesis.

## Workflow Architecture

<div align="center">
  <img src="../images/workflow_adaptive_query_routing.png" alt="Adaptive Query Routing Architecture Diagram" width="680" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["Query Ingestion"]
        Q(["1. User Query"]):::startNode
    end
    subgraph Decision["Runtime Router Engine"]
        Router{"2. Intent & Pattern Classifier<br/><b>Heuristics + Groq LLM</b>"}:::routerNode
    end
    subgraph Engines["Specialized Retrieval Engines"]
        BM25["3a. BM25 Lexical Engine<br/>(Exact IDs, Errors, Codes)"]:::bm25Node
        HyDE["3b. HyDE Embedding Engine<br/>(Abstract & Short Concepts)"]:::hydeNode
        Decomp["3c. Multi-Query Decomposer<br/>(Comparative & Multi-part)"]:::decompNode
        Hybrid["3d. Hybrid Engine<br/>(Dense + BM25 Default)"]:::hybridNode
    end
    subgraph Rerank["Fusion & Calibration"]
        Fuse["4. Context Assembler / Reranker"]:::fuseNode
    end
    subgraph Out["Grounded Synthesis"]
        Synth["5. LLM Answer Synthesizer<br/><b>Groq gpt-oss-120b</b>"]:::synthNode
        Ans(["6. Grounded Production Answer"]):::endNode
    end

    Q --> Router
    Router -->|"Exact match / Code"| BM25
    Router -->|"Abstract / Vague"| HyDE
    Router -->|"Comparison / Multi-hop"| Decomp
    Router -->|"Standard / General"| Hybrid
    BM25 --> Fuse
    HyDE --> Fuse
    Decomp --> Fuse
    Hybrid --> Fuse
    Fuse --> Synth
    Q --> Synth
    Synth --> Ans

    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef routerNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef bm25Node fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
    classDef hydeNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef decompNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef hybridNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef fuseNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef synthNode fill:#F3E5F5,stroke:#7B1FA2,stroke-width:2px,color:#4A148C;
    classDef endNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
```
</details>

### Runtime Decision Matrix Summary

| Strategy | Best Suited For | Typical Query Signs | Overhead & Latency |
| :--- | :--- | :--- | :--- |
| **BM25 Lexical** | Error codes, SKUs, API symbols | Alphanumeric tokens, CamelCase, quotes | < 5 ms (Ultra fast) |
| **HyDE** | Broad, short, abstract topics | < 4 tokens, vocabulary gap | ~200-400 ms (1 LLM call) |
| **Decomposition** | Multi-hop, comparisons | 'vs', 'difference between', multiple 'and' | ~300-600 ms (N sub-queries) |
| **Hybrid Search** | Standard open-domain Q&A | Natural language questions | ~15-30 ms (Vector + Lexical) |

## Step 1: Environment Setup & Dataset Preparation

We load our technical documentation dataset (`data/langchain_crewai_dataset.txt`), split it into manageable semantic chunks, and initialize the embedding model and Groq chat LLM.

In [1]:
import os
import re
from typing import List, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# 1. Initialize Embedding Model & Chat LLM
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = init_chat_model(model="groq:openai/gpt-oss-120b")

# 2. Load dataset and split into chunks
loader = TextLoader(os.path.join("..", "data", "langchain_crewai_dataset.txt") if os.path.exists(os.path.join("..", "data", "langchain_crewai_dataset.txt")) else os.path.join("data", "langchain_crewai_dataset.txt"))
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=60)
chunks = splitter.split_documents(raw_docs)
print(f"Loaded {len(raw_docs)} documents into {len(chunks)} text chunks.")

C:\Users\itsar\AppData\Local\Temp\ipykernel_30820\2793641180.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8810.66it/s]


Loaded 1 documents into 210 text chunks.


## Step 2: Building the Specialized Retrieval Engines

We configure distinct retrieval engines tailored to specific query characteristics:
1. **Dense Vector Engine (`Chroma`)**: High semantic awareness for conceptual similarity.
2. **BM25 Lexical Engine (`BM25Retriever`)**: Token-exact frequency matching for terms and IDs.
3. **HyDE Engine**: Hallucinates a speculative passage via LLM before embedding lookup.
4. **Hybrid Ensemble Engine**: Merges dense and lexical results with reciprocal ranking.

In [2]:
# 1. Dense Vector Store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="adaptive_dense_store"
)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 2. BM25 Lexical Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# 3. HyDE Generator Chain
hyde_prompt = PromptTemplate.from_template(
    "Write a concise, hypothetical technical passage answering this topic: {query}"
)
hyde_generator = hyde_prompt | llm

def retrieve_hyde(query: str, k: int = 4):
    hypo_doc = hyde_generator.invoke({"query": query}).content
    return vectorstore.similarity_search(hypo_doc, k=k)

# 4. Sub-Query Decomposer Chain
decomp_prompt = PromptTemplate.from_template(
    "Break down this comparative/multi-part question into 2 distinct, focused search queries.\n"
    "Provide each sub-query on a new line with no numbering.\n"
    "Question: {query}"
)
decomp_chain = decomp_prompt | llm

def retrieve_decomposed(query: str, k_each: int = 2):
    sub_queries = [q.strip() for q in decomp_chain.invoke({"query": query}).content.strip().split("\n") if q.strip()]
    aggregated_docs = []
    for sq in sub_queries[:2]:
        aggregated_docs.extend(vectorstore.similarity_search(sq, k=k_each))
    return aggregated_docs

# 5. Hybrid Retrieval (Dense + BM25 Fusion)
def retrieve_hybrid(query: str, k: int = 4):
    dense_hits = dense_retriever.invoke(query)
    bm25_hits = bm25_retriever.invoke(query)
    # Deduplicate preserving order
    seen = set()
    merged = []
    for doc in dense_hits + bm25_hits:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            merged.append(doc)
    return merged[:k]

print("All 4 specialized retrieval engines initialized successfully!")

All 4 specialized retrieval engines initialized successfully!


## Step 3: Designing the Runtime Adaptive Query Router

The router decides which engine to invoke based on two tiers:
- **Fast Heuristic Pass (0-1ms)**: Checks for exact code patterns, error keywords, or very short queries using Regex.
- **Semantic LLM Classifier Pass (~150ms)**: Uses structured JSON schema via `with_structured_output` to categorize complex, comparative, or conceptual queries.

In [3]:
class RouteDecision(BaseModel):
    """Schema for determining dynamic retrieval strategy."""
    strategy: Literal["bm25", "hyde", "decomposition", "hybrid"] = Field(
        description="The chosen retrieval strategy based on query analysis."
    )
    rationale: str = Field(
        description="Concise explanation for choosing this specific retrieval strategy."
    )

# LLM Router with Structured Output
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert Query Classifier in a production RAG system.
Classify the incoming query into exactly ONE strategy:
- 'bm25': Contains exact class names, methods, errors, or technical identifiers (e.g. ConversationBufferMemory, v0.2, error 404).
- 'hyde': Highly abstract, brief conceptual topic lacking technical keywords (e.g., 'agent synergy', 'human alignment').
- 'decomposition': Comparative questions comparing two distinct entities, or multi-step queries (e.g., 'Compare CrewAI vs LangGraph memory').
- 'hybrid': Standard natural language question with both semantic and keyword context.
"""),
    ("human", "{query}")
])

structured_router = router_prompt | llm.with_structured_output(RouteDecision)

def route_query(query: str) -> RouteDecision:
    # Heuristic fast-path 1: Comparison indicator keywords take highest precedence
    if any(kw in query.lower() for kw in ["compare", "difference between", "vs", "versus"]):
        return RouteDecision(
            strategy="decomposition",
            rationale="Fast-path heuristic detected explicit comparative syntax."
        )
    
    # Heuristic fast-path 2: Specific code class identifiers, errors, or version tags
    if re.search(r"(\b\w+(?:Memory|Retriever|Parser|Error|Exception)\b|error\s*\d+|v\d+\.\d+)", query, re.IGNORECASE):
        return RouteDecision(
            strategy="bm25",
            rationale="Fast-path heuristic detected specific technical class/error identifier."
        )
    
    # Fallback to LLM structured classifier
    return structured_router.invoke({"query": query})

print("Adaptive Query Router configured.")


Adaptive Query Router configured.


## Step 4: The End-to-End Adaptive RAG Pipeline

We now orchestrate the complete workflow: routing the query, invoking the selected retriever, assembling context, and executing grounded answer generation.

In [4]:
# RAG synthesis chain
rag_prompt = PromptTemplate.from_template("""
Answer the user query based ONLY on the provided context below.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

def adaptive_rag(query: str):
    print("=" * 75)
    print(f"USER QUERY: '{query}'")
    
    # 1. Dynamic Routing
    decision = route_query(query)
    print(f"[ROUTER DECISION] -> Strategy: {decision.strategy.upper()}")
    print(f"[ROUTER RATIONALE] -> {decision.rationale}")
    
    # 2. Strategy Execution
    if decision.strategy == "bm25":
        docs = bm25_retriever.invoke(query)
    elif decision.strategy == "hyde":
        docs = retrieve_hyde(query)
    elif decision.strategy == "decomposition":
        docs = retrieve_decomposed(query)
    else: # hybrid
        docs = retrieve_hybrid(query)
        
    print(f"[RETRIEVAL] Retrieved {len(docs)} documents.")
    
    # 3. Grounded Answer Synthesis
    answer = rag_chain.invoke({"input": query, "context": docs})
    print(f"\n[SYNTHESIZED ANSWER]:\n{answer}\n")
    return answer

## Step 5: Runtime Demonstrations Across Different Query Types

Let us test the adaptive router across 4 radically different queries to verify that each routes to its optimal retrieval engine.

In [5]:
test_queries = [
    # Test 1: Exact class identifier -> Routes to BM25
    "How does ConversationBufferMemory manage message histories?",
    
    # Test 2: Comparative question -> Routes to Decomposition
    "Compare hierarchical multi-agent teams vs single-agent sequential execution",
    
    # Test 3: Abstract conceptual topic -> Routes to HyDE
    "Autonomous team synergy",
    
    # Test 4: General natural language question -> Routes to Hybrid
    "What tools and actions can agents perform during task execution?"
]

for query in test_queries:
    adaptive_rag(query)


USER QUERY: 'How does ConversationBufferMemory manage message histories?'
[ROUTER DECISION] -> Strategy: BM25
[ROUTER RATIONALE] -> Fast-path heuristic detected specific technical class/error identifier.
[RETRIEVAL] Retrieved 4 documents.

[SYNTHESIZED ANSWER]:
ConversationBufferMemory keeps a running **buffer of the chat turns**.  
Each time a user or the assistant sends a message, that message is appended to the buffer, so the memory holds the entire sequence of prior exchanges. When the LLM is called, the buffer is passed in as the context, letting the model stay aware of everything that has been said so far. In short, ConversationBufferMemory — as a memory module—stores the full message history in order, continually adding new turns so the LLM can reference the conversation’s past.

USER QUERY: 'Compare hierarchical multi-agent teams vs single-agent sequential execution'
[ROUTER DECISION] -> Strategy: DECOMPOSITION
[ROUTER RATIONALE] -> Fast-path heuristic detected explicit compara